# T4x2 Concurrent Test - SANA-Sprint (cuda:0) + Chatterbox TTS (cuda:1)
Verify both GPUs are used simultaneously and compare concurrent vs sequential wall-clock.


In [ ]:
import os, sys, time, json, subprocess
from pathlib import Path
import torch
from kaggle import KaggleApi

api = KaggleApi(); api.authenticate()
n = torch.cuda.device_count()
names = [torch.cuda.get_device_name(i) for i in range(n)]
ccs = [torch.cuda.get_device_properties(i).major for i in range(n)]
print('device_count =', n, 'names =', names, 'ccs =', ccs)
if n != 2 or any(c < 7 for c in ccs):
    print(f'[FATAL] Expected 2x T4 (CC>=7); got device_count={n} ccs={ccs} names={names}. Aborting (retry for T4x2).')
    sys.exit(1)
print('2x T4 confirmed. Proceeding.')

def get_quota():
    for fn in (lambda: api.quota_view(), lambda: getattr(api, 'api', None) and api.api.quota_view()):
        try:
            q = fn()
            if q is not None:
                return {'gpu_quota': getattr(q, 'gpu_quota', None), 'quota_refresh_time': str(getattr(q, 'quota_refresh_time', None))}
        except Exception as e:
            print('  quota access failed:', repr(e))
    return None

quota_before = get_quota()
print('QUOTA BEFORE:', quota_before)


In [ ]:
import subprocess, sys
print('Installing chatterbox-tts (--no-deps, keep Kaggle numpy/torch ABI)...', flush=True)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'chatterbox-tts'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'resemble-perth>=1.0.0', 'conformer==0.3.2', 'spacy-pkuseg', 'pykakasi==2.3.0', 'pyloudnorm', 'omegaconf', 's3tokenizer', 'librosa==0.11.0', 'gradio==6.8.0'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'transformers==5.2.0', 'diffusers==0.29.0'])
print('Install done.', flush=True)


In [ ]:
import os, sys, time, json, subprocess
from pathlib import Path

IMG_OUT = Path('/kaggle/working/img_out'); IMG_OUT.mkdir(parents=True, exist_ok=True)
TTS_OUT = Path('/kaggle/working/tts_out'); TTS_OUT.mkdir(parents=True, exist_ok=True)
WORKERS = Path('/kaggle/working')

def run_worker(script, gpu):
    env = dict(os.environ); env['CUDA_VISIBLE_DEVICES'] = str(gpu)
    return subprocess.Popen([sys.executable, str(WORKERS / script)], env=env,
                             stdout=(WORKERS / f'{script}.out').open('w'),
                             stderr=subprocess.STDOUT)

# ---- CONCURRENT: image on cuda:0, tts on cuda:1, at the same time ----
p_img = run_worker('image_worker.py', 0)
p_tts = run_worker('tts_worker.py', 1)
print('Launched image_worker (cuda:0) + tts_worker (cuda:1) concurrently.', flush=True)

samples = []
t_start = time.time()
procs = [p_img, p_tts]
while any(p.poll() is None for p in procs):
    try:
        out = subprocess.run(['nvidia-smi', '--query-gpu=index,utilization.gpu,memory.used',
                              '--format=csv,noheader'], capture_output=True, text=True, timeout=15).stdout
        samples.append((round(time.time() - t_start, 1), out.rstrip('\n')))
    except Exception as e:
        samples.append((round(time.time() - t_start, 1), f'smi_err {e}'))
    time.sleep(2)
combined_s = round(time.time() - t_start, 2)
rc_img, rc_tts = p_img.wait(), p_tts.wait()
print(f'CONCURRENT done in {combined_s}s (img rc={rc_img}, tts rc={rc_tts})', flush=True)

# capture concurrent worker manifests before sequential run overwrites them
img_man = json.loads((IMG_OUT / 'worker_manifest.json').read_text())
tts_man = json.loads((TTS_OUT / 'worker_manifest.json').read_text())
print('IMG concurrent:', {k: img_man.get(k) for k in ('model_load_s', 'total_gen_s')})
print('TTS concurrent:', {k: tts_man.get(k) for k in ('model_load_s', 'total_gen_s', 'rtf')})
for s in samples[:3]: print('smi t=%s:\n%s' % s)
for s in samples[-3:]: print('smi t=%s:\n%s' % s)


In [ ]:
import os, sys, time, subprocess
from pathlib import Path

# ---- SEQUENTIAL baseline on a SINGLE T4 (cuda:0 for both, one after another) ----
def run_seq(script):
    env = dict(os.environ); env['CUDA_VISIBLE_DEVICES'] = '0'
    t0 = time.time()
    r = subprocess.run([sys.executable, str(Path('/kaggle/working') / script)], env=env,
                       stdout=(Path('/kaggle/working') / f'seq_{script}.out').open('w'), stderr=subprocess.STDOUT)
    return round(time.time() - t0, 2), r.returncode

seq_img_s, rc1 = run_seq('image_worker.py')
print(f'SEQUENTIAL image (cuda:0) wall={seq_img_s}s rc={rc1}', flush=True)
seq_tts_s, rc2 = run_seq('tts_worker.py')
print(f'SEQUENTIAL tts (cuda:0) wall={seq_tts_s}s rc={rc2}', flush=True)
seq_total_s = round(seq_img_s + seq_tts_s, 2)
print(f'SEQUENTIAL total = {seq_total_s}s', flush=True)


In [ ]:
import json, time
from kaggle import KaggleApi
from pathlib import Path

api = KaggleApi(); api.authenticate()
quota_after = get_quota()
print('QUOTA AFTER:', quota_after)

# both GPUs utilized? check any sample where more than one gpu index shows >0 util
both_util = False
for _, raw in samples:
    lines = [l for l in raw.splitlines() if l.strip()]
    non_zero = [l for l in lines if ', 0 %' not in l and 'utilization.gpu' not in l and l.strip()]
    # each data line looks like '0, 35 %, 1234 MiB'; >1 lines with MiB => >1 gpu active
    if len(non_zero) >= 2:
        both_util = True
        break

master = {
    'machine_shape': '2 x NvidiaTeslaT4',
    'device_count': int(torch.cuda.device_count()),
    'gpu_names': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    'quota_before': quota_before,
    'quota_after': quota_after,
    'concurrent': {
        'wall_s': combined_s,
        'img_model_load_s': img_man.get('model_load_s'),
        'img_total_gen_s': img_man.get('total_gen_s'),
        'tts_model_load_s': tts_man.get('model_load_s'),
        'tts_total_gen_s': tts_man.get('total_gen_s'),
        'tts_rtf': tts_man.get('rtf'),
        'img_rc': rc_img, 'tts_rc': rc_tts,
    },
    'sequential': {
        'img_wall_s': seq_img_s,
        'tts_wall_s': seq_tts_s,
        'total_s': seq_total_s,
    },
    'speedup_x': round(seq_total_s / max(combined_s, 1e-6), 3),
    'both_gpus_utilized': both_util,
    'nvidia_smi_samples': [{'t': t, 'raw': raw} for t, raw in samples],
}
Path('/kaggle/working/master_manifest.json').write_text(json.dumps(master, indent=2))
print('SPEEDUP (seq/concurrent):', master['speedup_x'])
print('BOTH GPUS UTILIZED:', both_util)
print(json.dumps(master, indent=2))
